# makemore part 4: becoming a backprop ninja

This is the *Practice* step of `unit_05_makemore_backprop_ninja.md`. Do the Cold Attempt there first.

The forward pass is given, one named tensor per elementary step. You write the backward pass:
every gradient, by hand, at the tensor level, and it has to match autograd. Work top to bottom.
Each milestone is one cell of stubs followed by a grader cell. The grader stops at your first
failure so there is always exactly one thing in front of you.

**Rules of engagement**
- Don't open the lecture. Don't open the reference notebook.
- Stuck on an *idea* for 20 min → ask the coaching chat for a hint.
- Stuck on *PyTorch syntax* → ask immediately, zero learning value in that.
- **Before you run a grader cell, say out loud what you expect to happen.**

Each milestone is a function that takes the forward-pass namespace `t` and the dict of gradients
`g` you have built so far, adds the next few, and returns `g`. That is just so you build the
backward pass one cell at a time instead of re-running one giant cell.

In [ ]:
import torch
import torch.nn.functional as F
from test_makemore_backprop_ninja import grade, cmp, forward, init_params, make_batch, fresh_forward

## The forward pass (given)

Read this cell carefully; you will be walking it backwards. Every intermediate has a name and
`retain_grad()`, and `t.<name>.grad` is autograd's answer for that tensor, so you can check
yourself at any point with `cmp('dprobs', g['dprobs'], t.probs)`.

An identical copy of this function lives in the grader, so grading never depends on this cell.
`t` also carries the parameters (`t.C`, `t.W1`, `t.b1`, `t.bngain`, `t.bnbias`, `t.W2`, `t.b2`),
the batch (`t.Xb`, `t.Yb`) and `t.n`, the batch size.

In [ ]:
def forward_pass(params, Xb, Yb):
    # this is exactly what forward() in the grader computes. shown here so you can read it.
    C, W1, b1, bngain, bnbias, W2, b2 = params
    n = Xb.shape[0]
    emb = C[Xb]                                        # (n, 3, 10)
    embcat = emb.view(emb.shape[0], -1)                # (n, 30)
    # linear 1
    hprebn = embcat @ W1 + b1                          # (n, 64)
    # batchnorm
    bnmeani = 1 / n * hprebn.sum(0, keepdim=True)      # (1, 64)
    bndiff = hprebn - bnmeani                          # (n, 64)
    bndiff2 = bndiff ** 2                              # (n, 64)
    bnvar = 1 / (n - 1) * bndiff2.sum(0, keepdim=True) # (1, 64)   Bessel's correction
    bnvar_inv = (bnvar + 1e-5) ** -0.5                 # (1, 64)
    bnraw = bndiff * bnvar_inv                         # (n, 64)
    hpreact = bngain * bnraw + bnbias                  # (n, 64)
    # nonlinearity
    h = torch.tanh(hpreact)                            # (n, 64)
    # linear 2
    logits = h @ W2 + b2                               # (n, 27)
    # cross entropy, spelled out
    logit_maxes = logits.max(1, keepdim=True).values   # (n, 1)
    norm_logits = logits - logit_maxes                 # (n, 27)
    counts = norm_logits.exp()                         # (n, 27)
    counts_sum = counts.sum(1, keepdim=True)           # (n, 1)
    counts_sum_inv = counts_sum ** -1                  # (n, 1)
    probs = counts * counts_sum_inv                    # (n, 27)
    logprobs = probs.log()                             # (n, 27)
    loss = -logprobs[range(n), Yb].mean()              # ()
    return loss

# the real thing: a seeded batch of 32, autograd already run
t = fresh_forward()
print('loss', t.loss.item())
print('hprebn', tuple(t.hprebn.shape), '| autograd dhprebn', tuple(t.hprebn.grad.shape))
print('bnmeani', tuple(t.bnmeani.shape), '| b1', tuple(t.b1.shape), '| C', tuple(t.C.shape))

## Milestone 1 — from the loss to `counts`

Five tensors: `dlogprobs`, `dprobs`, `dcounts_sum_inv`, `dcounts_sum`, `dcounts`.
Every gradient has the same shape as the tensor it belongs to. Before writing each one, look at
the forward line that *produced* the tensor below it, and at every forward line that *used* it.

In [ ]:
def backward_1(t):
    """Start the backward pass at the loss.

    t: forward namespace (see above). Use t.n, t.Yb, t.logprobs, t.probs, t.counts,
       t.counts_sum, t.counts_sum_inv.
    Returns a dict g with keys
      dlogprobs        (n, 27)   dloss/dlogprobs
      dprobs           (n, 27)
      dcounts_sum_inv  (n, 1)
      dcounts_sum      (n, 1)
      dcounts          (n, 27)   ALL of dloss/dcounts, every path included
    Do not touch autograd (.grad, .backward) except to check yourself with cmp().
    """
    raise NotImplementedError

In [ ]:
grade(backward_1, upto=1)

## Milestone 2 — the softmax: `norm_logits`, `logit_maxes`, `logits`

Three tensors. One of them is a gradient into something that, mathematically, should not
affect the loss at all. Predict what its gradient is *before* you compute it, then look at
what you get and explain the difference.

In [ ]:
def backward_2(t, g):
    """Continue from g (the dict backward_1 returned). Add
      dnorm_logits  (n, 27)
      dlogit_maxes  (n, 1)
      dlogits       (n, 27)   every path included
    and return g.
    """
    raise NotImplementedError

In [ ]:
grade(backward_1, backward_2, upto=2)

## Milestone 3 — the second linear layer and tanh

`logits = h @ W2 + b2` and `h = tanh(hpreact)`. Four tensors. For the matmul, get the shapes
to line up first and the answer is nearly forced; then check the rule on a 2x2 by hand if you
don't trust it.

In [ ]:
def backward_3(t, g):
    """Add
      dh        (n, 64)
      dW2       (64, 27)
      db2       (27,)
      dhpreact  (n, 64)
    and return g. Use t.h, t.W2 and what is already in g.
    """
    raise NotImplementedError

In [ ]:
grade(backward_1, backward_2, backward_3, upto=3)

## Milestone 4 — the batchnorm chain

`backward_4(t, g)` adds nine tensors to `g` and returns `g`, earlier keys untouched. Each key is
`dloss/d<name>` for the forward tensor of the same name and has exactly that tensor's shape.
Available: `g['dhpreact']` plus `t.bngain, t.bnraw, t.bnvar_inv, t.bndiff, t.bnvar, t.bndiff2,
t.hprebn, t.n`. `cmp('dbnraw', g['dbnraw'], t.bnraw)` checks any one of them against autograd.

| key | shape | gradient w.r.t. | forward line that consumes it |
|---|---|---|---|
| `dbngain` | `(1, 64)` | `bngain` | `hpreact = bngain * bnraw + bnbias` |
| `dbnraw` | `(n, 64)` | `bnraw` | same |
| `dbnbias` | `(1, 64)` | `bnbias` | same |
| `dbnvar_inv` | `(1, 64)` | `bnvar_inv` | `bnraw = bndiff * bnvar_inv` |
| `dbnvar` | `(1, 64)` | `bnvar` | `bnvar_inv = (bnvar + 1e-5) ** -0.5` |
| `dbndiff2` | `(n, 64)` | `bndiff2` | `bnvar = 1 / (n - 1) * bndiff2.sum(0, keepdim=True)` |
| `dbndiff` | `(n, 64)` | `bndiff` | two consumers: `bnraw = bndiff * bnvar_inv` and `bndiff2 = bndiff ** 2`; the total |
| `dbnmeani` | `(1, 64)` | `bnmeani` | `bndiff = hprebn - bnmeani` |
| `dhprebn` | `(n, 64)` | `hprebn` | two consumers: `bndiff = hprebn - bnmeani` and `bnmeani = 1 / n * hprebn.sum(0, keepdim=True)`; the total |

A missing key, a wrong shape, or values off from autograd (rtol 1e-5, atol 1e-7) fails.

The grader checks, in order: dbngain → dbnraw → dbnbias → dbnvar_inv → dbnvar → dbndiff2 →
dbndiff → dbnmeani → dhprebn. Add one, re-run, repeat.

In [ ]:
def backward_4(t, g):
    """Add, in this order,
      dbngain    (1, 64)
      dbnraw     (n, 64)
      dbnbias    (1, 64)
      dbnvar_inv (1, 64)
      dbnvar     (1, 64)
      dbndiff2   (n, 64)
      dbndiff    (n, 64)   every path included (there are two)
      dbnmeani   (1, 64)
      dhprebn    (n, 64)   every path included (there are two)
    and return g. Use t.bngain, t.bnraw, t.bnvar_inv, t.bndiff, t.bnvar, t.bndiff2,
    t.hprebn, t.n.
    """
    raise NotImplementedError

In [ ]:
grade(backward_1, backward_2, backward_3, backward_4, upto=4)

## Milestone 5 — the first linear layer and the embedding lookup

`backward_5(t, g)` adds five tensors to `g` and returns `g`. Available: `g['dhprebn']` plus
`t.embcat, t.W1, t.emb, t.C, t.Xb`.

| key | shape | gradient w.r.t. | forward line that consumes it |
|---|---|---|---|
| `dembcat` | `(n, 30)` | `embcat` | `hprebn = embcat @ W1 + b1` |
| `dW1` | `(30, 64)` | `W1` | same |
| `db1` | `(64,)` | `b1` | same. Shape is `(64,)`, not `(1, 64)` |
| `demb` | `(n, 3, 10)` | `emb` | `embcat = emb.view(n, -1)` |
| `dC` | `(27, 10)` | `C` | `emb = C[Xb]`. A row of `C` never looked up in this batch is all zeros; a row looked up k times holds the total over all k lookups |

The grader checks, in order: dembcat → dW1 → db1 → demb → dC.

In [ ]:
def backward_5(t, g):
    """Add
      dembcat  (n, 30)
      dW1      (30, 64)
      db1      (64,)
      demb     (n, 3, 10)
      dC       (27, 10)
    and return g. Use t.embcat, t.W1, t.emb, t.C, t.Xb.
    """
    raise NotImplementedError

In [ ]:
grade(backward_1, backward_2, backward_3, backward_4, backward_5, upto=5)

## Milestone 6 (stretch) — the fused shortcuts

Two standalone functions. Neither reads `g`. The grader calls each on the seeded batch of 32 and
again on a fresh batch of 48 (seed 7), so nothing may assume `n == 32`.

**`dlogits_fast(t)`**
- Inputs: `t.logits (n, 27)`, `t.Yb (n,)`, `t.n`. Nothing from milestones 1–2 (no `counts`,
  `probs`, `logit_maxes`, no `g`).
- Returns `dloss/dlogits`, shape `(n, 27)`, matching autograd's `t.logits.grad`
  (rtol 1e-5, atol 1e-7), as one short expression.

**`dhprebn_fast(t, dhpreact)`**
- Inputs: `dhpreact (n, 64)` (the grader passes autograd's own `t.hpreact.grad`, so this check
  does not depend on your milestone 3), plus `t.bngain (1, 64)`, `t.bnvar_inv (1, 64)`,
  `t.bnraw (n, 64)`, `t.n`.
- Returns `dloss/dhprebn`, shape `(n, 64)`, matching autograd's `t.hprebn.grad`, as ONE
  expression. None of the nine milestone-4 intermediates may appear.

The grader checks, in order: dlogits_fast on batch 32 → dhprebn_fast on batch 32 →
dlogits_fast on batch 48 → dhprebn_fast on batch 48.

In [ ]:
def dlogits_fast(t):
    """dloss/dlogits in one short expression, straight from t.logits, t.Yb, t.n.
    Returns (n, 27). Must not use anything from g."""
    raise NotImplementedError


def dhprebn_fast(t, dhpreact):
    """dloss/dhprebn in ONE expression, given dhpreact (n, 64) and t.bngain, t.bnvar_inv,
    t.bnraw, t.n. Returns (n, 64). No intermediate d-tensors from milestone 4."""
    raise NotImplementedError

In [ ]:
grade(backward_1, backward_2, backward_3, backward_4, backward_5, dlogits_fast, dhprebn_fast, upto=6)

## Milestone 7 (stretch) — train with only your gradients

**`param_grads(t)`**
- Input: `t` from `forward(params, Xb, Yb, autograd=False)`. Every intermediate and parameter is
  present by name, no tensor has a `.grad`, and nothing inside may call `.backward()` or read a
  `.grad`.
- Returns a list (or tuple) of exactly 7 tensors, in this order and with these shapes:
  `dC (27, 10)`, `dW1 (30, 64)`, `db1 (64,)`, `dbngain (1, 64)`, `dbnbias (1, 64)`,
  `dW2 (64, 27)`, `db2 (27,)`. Each is `dloss/d<param>` for the batch inside `t`.
- Milestone 6 is not required: `param_grads` can be composed from `backward_1..5` alone. If it
  does call `dlogits_fast` / `dhprebn_fast`, those must exist even though the grade cell below
  skips milestone 6.

The grader checks, in order: returns a 7-list → dC → dW1 → db1 → dbngain → dbnbias → dW2 → db2
(each against autograd on the seeded batch) → 300 steps of `p -= 0.1 * dp` from
`init_params(seed=1)`, batch 32, `autograd=False` forwards, after which the loss on the first
4000 training rows must be below 2.75.

The grade cell below passes `skip=(6,)`: nothing later requires `dlogits_fast` or `dhprebn_fast`.
Remove `skip` to grade milestone 6 as well.

In [ ]:
def param_grads(t):
    """Return [dC, dW1, db1, dbngain, dbnbias, dW2, db2], each the same shape as its
    parameter, computed by hand from t. t may come from forward(..., autograd=False),
    so nothing in here may read a .grad or call .backward()."""
    raise NotImplementedError

In [ ]:
grade(backward_1, backward_2, backward_3, backward_4, backward_5, dlogits_fast, dhprebn_fast, param_grads, skip=(6,))

## Your own training loop

The grader trained the MLP for you in milestone 7. Now write the loop yourself, from memory,
with no autograd anywhere: forward (`autograd=False`), your `param_grads`, the update. Print
the loss every 50 steps and then sample a few names from the trained model (write the
sampling loop too: context of 3, feed through the same forward math, sample from `probs`,
stop at `.`). Note where the batchnorm statistics come from at sampling time and what
that means for a batch of one.

In [ ]:
params = init_params(seed=3)
for step in range(300):
    Xb, Yb = make_batch(seed=step, batch_size=32)
    ...

## Scratch

`t = fresh_forward()` gives you a fresh seeded batch with autograd already run.
`cmp('dprobs', g['dprobs'], t.probs)` prints exact / approximate / max diff against autograd.